In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc matplotlib numpy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# Estimarea energiei stării fundamentale a lanțului Heisenberg cu VQE

import TutorialFeedback from '@site/src/components/TutorialFeedback';





*Estimare de utilizare: 37 de minute pe un procesor Heron (NOTĂ: Aceasta este doar o estimare. Timpul tău de execuție poate varia.)*
## Rezultate ale învățării
După finalizarea acestui tutorial, te poți aștepta să înțelegi următoarele informații:
- Cum să modelezi un lanț de spinuri Heisenberg ca Hamiltonian cuantic folosind Qiskit
- Cum să folosești optimizatorul SPSA pentru a estima energia stării fundamentale a unui sistem cuantic
- Cum să execuți fluxuri de lucru variaționali pe hardware cuantic IBM&reg; folosind primitivele și sesiunile Qiskit Runtime

## Cerințe prealabile
Se recomandă să te familiarizezi cu aceste subiecte:
- [Bazele informației cuantice](/learning/courses/basics-of-quantum-information)
- [Introducere în tiparele Qiskit](/guides/intro-to-patterns)
- [Designul algoritmilor variaționali](/learning/courses/variational-algorithm-design)

## Context
Lanțul de spinuri Heisenberg este unul dintre cele mai studiate modele din fizica materiei condensate și magnetismul cuantic. Descrie o rețea unidimensională de spinuri cuantice interacționante, unde spinurile vecine sunt cuplate prin interacțiuni de schimb. Hamiltonianul pentru modelul Heisenberg izotrop cu un câmp magnetic extern este dat de:

$$H = \sum_{\langle i,j \rangle} \left( J_x X_i X_j + J_y Y_i Y_j + J_z Z_i Z_j \right) + \sum_{i} h_i Z_i,$$

unde $X_i$, $Y_i$ și $Z_i$ sunt operatorii Pauli care acționează pe site-ul $i$, suma $\langle i,j \rangle$ rulează peste perechile de vecini apropiați, $J_x = J_y = J_z = 0.5$ sunt constantele de cuplare de schimb (izotrope în acest tutorial), iar $h_i$ reprezintă un câmp magnetic extern dependent de site. În acest tutorial, valorile câmpului magnetic sunt eșantionate aleator din intervalul $[-1, 1]$. Rețineți că în implementarea de mai jos, setul de perechi „vecini apropiați" este determinat de cuplarea nativă a hardware-ului backend între primii $N$ qubiți, care ar putea să nu formeze un lanț liniar strict în funcție de topologia dispozitivului.

Înțelegerea energiei stării fundamentale a acestui Hamiltonian este de o importanță fundamentală în fizică. Starea fundamentală codifică informații despre tranzițiile de fază cuantice, structura entanglementului și ordonarea magnetică. Clasic, calculul exact al energiei stării fundamentale devine intractabil pe măsură ce numărul de spinuri crește, deoarece dimensiunea spațiului Hilbert scalează exponențial ca $2^N$ pentru $N$ spinuri. Aceasta îl face un candidat natural pentru simularea cuantică.

Eigensolver-ul Cuantic Variațional (VQE) este un algoritm hibrid cuantic-clasic conceput pentru a estima energia stării fundamentale a unui Hamiltonian. Funcționează prin pregătirea unei stări cuantice parametrizate $|\psi(\theta)\rangle$ (numită ansatz) pe un calculator cuantic și măsurarea valorii de așteptare $\langle \psi(\theta) | H | \psi(\theta) \rangle$. Un optimizator clasic ajustează apoi iterativ parametrii $\theta$ pentru a minimiza această energie, valorificând principiul variațional care garantează că energia măsurată este întotdeauna o limită superioară pentru energia adevărată a stării fundamentale.

În acest tutorial, folosim ansatz-ul `efficient_su2` din biblioteca de circuite Qiskit, care construiește straturi de rotații pe un singur qubit și porți de entanglement. Optimizarea este realizată folosind algoritmul Simultaneous Perturbation Stochastic Approximation (SPSA), care este bine adaptat pentru hardware cuantic zgomotos deoarece estimează gradienții folosind doar două evaluări de funcție pe iterație, indiferent de numărul de parametri.
## Cerințe
Înainte de a începe acest tutorial, asigură-te că ai instalate următoarele:

* Qiskit SDK v2.0 sau o versiune mai recentă, cu suport pentru [vizualizare](https://docs.quantum.ibm.com/api/qiskit/visualization)
* Qiskit Runtime v0.44 sau o versiune mai recentă (`pip install qiskit-ibm-runtime`)
## Configurare

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from typing import Sequence

from qiskit import QuantumCircuit
from qiskit.quantum_info import SparsePauliOp
from qiskit.primitives import BaseEstimatorV2
from qiskit.circuit.library import XGate
from qiskit.circuit.library import efficient_su2
from qiskit.transpiler import PassManager
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit.transpiler.passes.scheduling import (
    ALAPScheduleAnalysis,
    PadDynamicalDecoupling,
)
from qiskit_ibm_runtime import QiskitRuntimeService, Session, EstimatorV2


def visualize_results(results):
    plt.plot(results["cost_history"], lw=2)
    plt.xlabel("Number of function evaluations")
    plt.ylabel("Energy")
    plt.show()

## Exemplu la scară mică

În această secțiune, parcurgem fiecare pas al tiparului Qiskit la scară mică, explicând componentele cheie pe măsură ce construim fluxul de lucru.

### Pasul 1: Maparea intrărilor clasice la o problemă cuantică

* Intrare: Numărul de spinuri
* Ieșire: Ansatz și Hamiltonian care modelează lanțul Heisenberg

Construiește un ansatz și un Hamiltonian care modelează un lanț Heisenberg cu 10 spinuri. În acest pas, vom construi un Hamiltonian Heisenberg cu 10 spinuri peste harta de cuplare a backend-ului cel mai puțin aglomerat și vom pregăti ansatz-ul `efficient_su2`.

In [2]:
num_spins = 10
ansatz = efficient_su2(num_qubits=num_spins, reps=2)

service = QiskitRuntimeService()
backend = service.least_busy(
    operational=True, min_num_qubits=num_spins, simulator=False
)

coupling = backend.target.build_coupling_map()
reduced_coupling = coupling.reduce(list(range(num_spins)))

edge_list = reduced_coupling.graph.edge_list()
ham_list = []

for edge in edge_list:
    ham_list.append(("ZZ", edge, 0.5))
    ham_list.append(("YY", edge, 0.5))
    ham_list.append(("XX", edge, 0.5))

for qubit in reduced_coupling.physical_qubits:
    ham_list.append(("Z", [qubit], np.random.random() * 2 - 1))

hamiltonian = SparsePauliOp.from_sparse_list(ham_list, num_qubits=num_spins)

ansatz.draw("mpl", style="iqp")

<Image src="../docs/images/tutorials/spin-chain-vqe/extracted-outputs/7e8d2f10-f1d6-4ec2-bac9-9db23499c9e1-0.avif" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/tutorials/spin-chain-vqe/extracted-outputs/7e8d2f10-f1d6-4ec2-bac9-9db23499c9e1-0.avif)

### Pasul 2: Optimizarea problemei pentru execuția pe hardware cuantic
* Intrare: Circuit abstract, observabilă
* Ieșire: Circuit și observabilă țintă, optimizate pentru QPU selectat

Folosește funcția `generate_preset_pass_manager` din Qiskit pentru a genera automat o rutină de optimizare pentru circuitul nostru în raport cu QPU selectat. Alegem `optimization_level=3`, care oferă cel mai înalt nivel de optimizare al managerilor de pase presetate. Includem, de asemenea, pase de planificare `ALAPScheduleAnalysis` și `PadDynamicalDecoupling` pentru a suprima erorile de decoerență.

In [3]:
target = backend.target
pm = generate_preset_pass_manager(optimization_level=3, target=target)
pm.scheduling = PassManager(
    [
        ALAPScheduleAnalysis(durations=target.durations()),
        PadDynamicalDecoupling(
            durations=target.durations(),
            dd_sequence=[XGate(), XGate()],
            pulse_alignment=target.pulse_alignment,
        ),
    ]
)
isa_ansatz = pm.run(ansatz)
isa_observable = hamiltonian.apply_layout(isa_ansatz.layout)
isa_ansatz.draw("mpl", scale=0.6, style="iqp", fold=-1, idle_wires=False)

<Image src="../docs/images/tutorials/spin-chain-vqe/extracted-outputs/a0a5f1c8-5c31-4d9f-ae81-37bd67271d44-0.avif" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/tutorials/spin-chain-vqe/extracted-outputs/a0a5f1c8-5c31-4d9f-ae81-37bd67271d44-0.avif)

### Pasul 3: Execuția folosind primitivele Qiskit
* Intrare: Circuit și observabilă țintă
* Ieșire: Rezultatele optimizării

Minimizează energia estimată a stării fundamentale a sistemului prin optimizarea parametrilor circuitului. Folosește primitiva `Estimator` din Qiskit Runtime pentru a evalua funcția de cost în timpul optimizării.

Deoarece am optimizat circuitul pentru backend în Pasul 2, putem evita transpilarea pe serverul Runtime setând `skip_transpilation=True` și transmițând circuitul optimizat. Pentru această demonstrație, vom rula pe un QPU folosind primitivele `qiskit-ibm-runtime`. Pentru a rula cu primitivele bazate pe statevector din `qiskit`, înlocuiește blocul de cod care folosește primitivele Qiskit Runtime cu blocul comentat.

În acest tutorial folosim Simultaneous Perturbation Stochastic Approximation (SPSA), care este un optimizator bazat pe gradient. În continuare oferim o scurtă introducere în el și codul pentru a implementa SPSA folosind Qiskit v2.0.
### Introducere în SPSA
Simultaneous Perturbation Stochastic Approximation (SPSA) [\[1\]](#references) este un algoritm de optimizare care aproximează întregul vector gradient folosind doar două apeluri de funcție la fiecare iterație. Fie $f:\mathbb{R}^p\rightarrow \mathbb{R}$ funcția de cost cu $p$ parametri de optimizat, și $x_i\in \mathbb{R}^p$ vectorul de parametri la pasul $i^{th}$ al iterației. Pentru a calcula gradientul, se creează un vector aleator $\Delta_i$ de dimensiune $p$, unde fiecare element $\Delta_{ij}$, $\forall$ $j\in {1,2,...,p}$, este eșantionat uniform din ${-1, 1}$. Apoi, fiecare element al vectorului aleator $\Delta_i$ este înmulțit cu o valoare mică $c_i$ pentru a crea o perturbare aleatoare. Gradientul este estimat astfel:

$$[\nabla f(x_i)]_j \approx \frac{f(x_i + c_i \Delta_i) - f(x_i - c_i \Delta_i)}{2c_i\Delta_{ij}}.$$

Intuitiv, deoarece o perturbare aleatoare este aplicată în timpul estimării gradientului, se preconizează că mici deviații în valorile exacte ale $f$ provenite din zgomot pot fi tolerate și compensate. De fapt, SPSA este cunoscut în special pentru robustețea sa față de zgomot și necesită doar două apeluri hardware pentru fiecare iterație. Prin urmare, este unul dintre optimizatoarele preferate pentru implementarea algoritmilor variaționali.

În acest tutorial, hiperparametrii pentru iterația $i^{th}$, $a_i$ și $c_i$, sunt calculați ca

$$a_i = \frac{a}{(A + i + 1)^\alpha} \quad \text{and} \quad c_i = \frac{c}{(i+1)^\gamma},$$

unde valorile constante sunt $A = 30$, $\alpha = 0.9$, $a = 0.3$, $c = 0.1$ și $\gamma = 0.4$. Aceste valori sunt selectate din [\[2\]](#references). Ajustarea adecvată a hiperparametrilor este necesară pentru a extrage o performanță bună din SPSA.

In [4]:
def spsa(
    fun, x0, args=(), A=30, alpha=0.9, a=0.3, c=0.1, gamma=0.4, maxiter=100
):
    nparams = len(x0)
    x = np.copy(x0)

    for i in range(maxiter):
        a_i = a / (A + i + 1) ** alpha
        c_i = c / (i + 1) ** gamma
        delta_i = np.random.choice([-1, 1], nparams)

        # two hardware calls
        eval_1 = fun(x + c_i * delta_i, *args)
        eval_2 = fun(x - c_i * delta_i, *args)

        # compute the gradient and update the parameters
        grad = (eval_1 - eval_2) / (2 * c_i) * np.reciprocal(delta_i)
        x = x - a_i * grad

    return x

In [8]:
def cost_func(
    params: Sequence,
    ansatz: QuantumCircuit,
    hamiltonian: SparsePauliOp,
    estimator: BaseEstimatorV2,
    cost_history_dict: dict,
) -> float:
    """Ground state energy evaluation."""
    energy = (
        estimator.run([(ansatz, hamiltonian, [params])]).result()[0].data.evs
    )

    cost_history_dict["iters"] += 1
    cost_history_dict["prev_vector"] = list(params)
    cost_history_dict["cost_history"].append(float(energy[0]))

    print(
        f"Fx Iters. done: {cost_history_dict['iters']} [Current cost: {round(energy[0], 5)}]",
        end="\r",
    )

    return energy


def solve(x0, isa_ansatz, isa_observable, maxiter=150):
    cost_history_dict = {
        "prev_vector": None,
        "iters": 0,
        "cost_history": [],
        "y_min": None,
    }

    # Evaluate the problem using a QPU via Qiskit IBM Runtime
    with Session(backend=backend) as session:
        estimator = EstimatorV2(mode=session)
        estimator.skip_transpilation = True
        estimator.options.environment.job_tags = ["TUT_HSVQE"]
        x_opt = spsa(
            cost_func,
            x0=x0,
            args=(isa_ansatz, isa_observable, estimator, cost_history_dict),
            maxiter=maxiter,
        )

        y_min = cost_func(
            x_opt, isa_ansatz, isa_observable, estimator, cost_history_dict
        )

    return y_min, cost_history_dict

In [9]:
np.random.seed(42)
num_params = ansatz.num_parameters
params = 2 * np.pi * np.random.random(num_params)

Aici setăm `maxiter = 50`. Rețineți că deoarece fiecare iterație necesită două apeluri la funcție pentru a calcula gradientul, numărul total de apeluri de funcție va fi $2 \times \text{maxiter}$. `maxiter` poate fi crescut la orice valoare mai mare pentru o estimare mai bună a energiei.

In [10]:
maxiter = 50
spsa_min, spsa_history = solve(
    params, isa_ansatz, isa_observable, maxiter=maxiter
)

Fx Iters. done: 101 [Current cost: -3.03843]

### Step 4: Post-process and return result in desired classical format

* Input: Ground-state energy estimates during optimization
* Output: Estimated ground-state energy

In [11]:
print(f"Estimated ground state energy: {spsa_min}")

Estimated ground state energy: [-3.03842968]


In [12]:
results = {
    "spsa": spsa_history,
}

visualize_results(spsa_history)

<Image src="../docs/images/tutorials/spin-chain-vqe/extracted-outputs/ecd7762a-0.avif" alt="Output of the previous code cell" />

## Large-scale hardware example

A large-scale hardware example is not included in this tutorial. As the number of qubits increases, VQE encounters significant challenges due to the [barren plateau](/learning/courses/variational-algorithm-design/optimization-loops#barren-plateaus) phenomenon: the gradient of the cost function vanishes exponentially with system size, making optimization practically infeasible for large circuits. Combined with hardware noise, this means that scaling VQE to larger spin chains does not produce reliably reproducible results. For approaches that overcome these limitations, see the Next Steps section below.

## Challenge

Now that you have a working VQE implementation for the Heisenberg chain, try the following:

1. **Experiment with ansatz depth:** Modify the `reps` parameter in `efficient_su2` (for example, try `reps=1` and `reps=3`). How does ansatz depth affect the estimated ground-state energy and convergence speed? At what point do you observe diminishing returns or instability?
2. **Tune SPSA hyperparameters:** Adjust the learning rate schedule parameters (`a`, `c`, `alpha`, `gamma`, `A`) and observe how they impact convergence. Can you find a configuration that converges faster than the defaults used here?
3. **Compare coupling topologies:** Instead of using the backend's native coupling map, try constructing a simple nearest-neighbor linear chain and compare the results. How does the connectivity of the physical hardware affect the transpiled circuit depth and final energy estimate?

## References

\[1] Spall, J. C. (2002). Implementation of the simultaneous perturbation algorithm for stochastic optimization.
IEEE Transactions on Aerospace and Electronic Systems, 34(3), 817-823.

\[2] Sahin, M. Emre, et al. (2025). Qiskit Machine Learning: an open-source library for quantum machine learning tasks at scale on quantum hardware and classical simulators. arXiv:2505.17756.

## Next steps

<Admonition type="tip" title="Recommendations">
  If you found this work interesting, you might be interested in the following material:

  * **Try Sample-based Quantum Diagonalization (SQD):** As demonstrated in this tutorial, VQE faces challenges at scale due to barren plateaus and high measurement overhead. IBM has developed [Sample-based Quantum Diagonalization (SQD)](/docs/guides/qiskit-addons-sqd) as a more scalable alternative. Unlike VQE, SQD avoids variational optimization entirely; instead, a quantum computer generates samples and a classical computer projects the Hamiltonian onto a subspace spanned by those samples and diagonalizes it. This provides an upper bound to the ground-state energy with significantly fewer measurements and without susceptibility to barren plateaus. Follow the [SQD tutorial](/docs/tutorials/sample-based-quantum-diagonalization) to see this approach in action.
  * **Explore the Quantum Diagonalization Algorithms course:** Deepen your understanding of both VQE and SQD, including their trade-offs, in the [Quantum diagonalization algorithms](/learning/courses/quantum-diagonalization-algorithms) course on IBM Quantum Learning.
</Admonition>